In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "generated_midis" / "baselines"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_sequences = np.load(PROCESSED_DIR / "train_sequences.npy", mmap_mode="r")
test_sequences = np.load(PROCESSED_DIR / "test_sequences.npy", mmap_mode="r")

print("Train shape:", train_sequences.shape)
print("Test shape:", test_sequences.shape)

In [ ]:
def sample_random_baseline(reference_sequence: np.ndarray) -> np.ndarray:
    """
    Random baseline:
    Preserve overall activation probability from the reference sequence.
    """
    p = float(reference_sequence.mean())
    random_roll = (np.random.rand(*reference_sequence.shape) < p).astype(np.float32)
    return random_roll

In [ ]:
def estimate_markov_probabilities(sequences: np.ndarray, max_sequences: int = 2000):
    """
    Estimate first-order transition probabilities per pitch:
      P(x_t = 1 | x_{t-1} = 0)
      P(x_t = 1 | x_{t-1} = 1)
    """
    sequences = np.array(sequences[:max_sequences], dtype=np.float32)

    prev = sequences[:, :-1, :]   # [N, T-1, F]
    curr = sequences[:, 1:, :]    # [N, T-1, F]

    prev0 = (prev == 0)
    prev1 = (prev == 1)

    eps = 1e-8

    p_on_given_off = (curr * prev0).sum(axis=(0, 1)) / (prev0.sum(axis=(0, 1)) + eps)
    p_on_given_on = (curr * prev1).sum(axis=(0, 1)) / (prev1.sum(axis=(0, 1)) + eps)

    return p_on_given_off.astype(np.float32), p_on_given_on.astype(np.float32)

In [ ]:
p_on_given_off, p_on_given_on = estimate_markov_probabilities(train_sequences, max_sequences=2000)
print("p_on_given_off shape:", p_on_given_off.shape)
print("p_on_given_on shape:", p_on_given_on.shape)

In [ ]:
def generate_markov_baseline(
    seed_frame: np.ndarray,
    seq_len: int,
    p_on_given_off: np.ndarray,
    p_on_given_on: np.ndarray,
) -> np.ndarray:
    """
    Generate a piano-roll sequence frame by frame.
    """
    pitch_dim = seed_frame.shape[0]
    out = np.zeros((seq_len, pitch_dim), dtype=np.float32)

    current = seed_frame.astype(np.float32).copy()
    out[0] = current

    for t in range(1, seq_len):
        probs = np.where(current > 0.5, p_on_given_on, p_on_given_off)
        current = (np.random.rand(pitch_dim) < probs).astype(np.float32)
        out[t] = current

    return out

In [ ]:
sample_idx = 0
reference = np.array(test_sequences[sample_idx], dtype=np.float32)

random_sample = sample_random_baseline(reference)
markov_sample = generate_markov_baseline(
    seed_frame=reference[0],
    seq_len=reference.shape[0],
    p_on_given_off=p_on_given_off,
    p_on_given_on=p_on_given_on,
)

print("Reference mean:", reference.mean())
print("Random mean:", random_sample.mean())
print("Markov mean:", markov_sample.mean())

In [ ]:
plt.figure(figsize=(12, 4))
plt.imshow(reference.T, aspect="auto", origin="lower")
plt.title("Reference Piano-Roll")
plt.xlabel("Time")
plt.ylabel("Pitch")
plt.show()

plt.figure(figsize=(12, 4))
plt.imshow(random_sample.T, aspect="auto", origin="lower")
plt.title("Random Baseline Piano-Roll")
plt.xlabel("Time")
plt.ylabel("Pitch")
plt.show()

plt.figure(figsize=(12, 4))
plt.imshow(markov_sample.T, aspect="auto", origin="lower")
plt.title("Markov Baseline Piano-Roll")
plt.xlabel("Time")
plt.ylabel("Pitch")
plt.show()

In [ ]:
from src.generation.midi_export import save_midi

save_midi(reference, OUTPUT_DIR / "baseline_reference.mid", threshold=0.5, debug=False)
save_midi(random_sample, OUTPUT_DIR / "baseline_random.mid", threshold=0.5, debug=False)
save_midi(markov_sample, OUTPUT_DIR / "baseline_markov.mid", threshold=0.5, debug=False)

print("Saved baseline MIDIs to:", OUTPUT_DIR)